# Figure 7 / Table 10 — error mitigation comparison (single-step MNIST QRU circuit)

Reads `figure7_table10_data.csv` (one row per method × observable × repetition, OTC operating point +0.8, observables Z0, Z4, Z6, X5, X7).

* Simulation and Qiskit Runtime rows: 3 repetitions each (ibm_marrakesh, 2026-05-04). Shaded band = min–max over repetitions, line = mean.
* QESEM: single run with the uncertainty returned by the service (`std`), drawn as points with error bars.
* PEC: 15 independent runs on ibm_phoenix (number of randomizations set to 400–1,600; 10–11 Oct 2026), drawn as the mean over the runs with the run-to-run standard deviation as error bars. The PEC rows are written once into the CSV by the cell "Replace the PEC rows" from `pec_fixedshots_results_ibm_phoenix.csv` (the previous CSV is kept as a backup).
* `sv_val` is the statevector reference of the same circuit and observable.

The settings cell holds all choices. The table cell recomputes Table 10 from the same file, so the table and the figure always come from identical numbers.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

CSV  = 'figure7_table10_data.csv'
OUT  = 'Figure7'                 # saves Figure7.png and Figure7.pdf
OBS  = ['Z0', 'Z4', 'Z6', 'X5', 'X7']

# method key -> (label, colour, kind); kind = 'ref' | 'band' | 'point'. Order = drawing order; set to None to hide.
STYLE = {
    'statevector':    ('Statevector (reference)',                 'black',   'ref'),
    'noise_model':    ('Noise model (ibm_marrakesh)',             '#7f7f7f', 'band'),
    'fake_marrakesh': ('FakeMarrakesh',                           '#bcbd22', 'band'),
    'baseline':       ('Readout mitigation only (ibm_marrakesh)', '#d62728', 'band'),
    'twirling':       ('Twirling (300 inst., ibm_marrakesh)',     '#9467bd', 'band'),
    'zne':            ('ZNE (32 twirling inst., ibm_marrakesh)',  '#ff7f0e', 'band'),
    'zne_twirling':   ('ZNE (300 twirling inst., ibm_marrakesh)', '#1f77b4', 'band'),
    'pec':            ('PEC (15 runs, ibm_phoenix)',              '#8c564b', 'point'),
    'qesem':          ('QESEM (ibm_marrakesh)',                   '#2ca02c', 'point'),
}
# POINT_OFFSET = {'qesem': -0.08, 'pec': +0.08}   # x-shift so the two error-bar series do not overlap
POINT_OFFSET = {'qesem': 0, 'pec': 0}   # x-shift so the two error-bar series do not overlap
# POINT_MARKER = {'qesem': 'o', 'pec': 's'}
MARKER = {'noise_model': 's', 'fake_marrakesh': 'o', 'twirling': '<', 'zne': '>',
          'baseline': '^', 'zne_twirling': '<', 'qesem': 'o', 'pec': 's'}

SHOW_LEGEND = True    # original Figure 7 has no legend; labels are added as text boxes afterwards
FIGSIZE, DPI = (10, 4.5), 300
YLIM = (-0.2, 1.0)

# PEC source: 15 runs on ibm_phoenix with >= 400 randomizations (written into CSV once, see next cell)
PEC_RESULTS = '../pec_phoenix/pec_fixedshots_results_ibm_phoenix.csv'
PEC_MIN_RAND = 400
# statevector reference of the PEC circuit (Table 10 group 0, rep 0), as used for the PEC deviations
PEC_SV = {'Z0': 0.7999, 'Z4': 0.8004, 'Z6': 0.7993, 'X5': 0.8003, 'X7': 0.7999}

In [ ]:
# ── Replace the PEC rows by the 15 ibm_phoenix runs (run once; safe to re-run) ──
import os, shutil
all_df = pd.read_csv(CSV)
old = all_df[all_df.method == 'pec']
if (old.get('device') == 'ibm_phoenix').all() and len(old) == 15 * len(OBS):
    print('PEC rows already replaced (15 runs, ibm_phoenix) - nothing to do')
else:
    backup = CSV.replace('.csv', '_before_phoenix.csv')
    if not os.path.exists(backup):
        shutil.copy(CSV, backup); print('backup written:', backup)
    r = pd.read_csv(PEC_RESULTS)
    r = r[r.num_randomizations_req >= PEC_MIN_RAND].sort_values(['num_randomizations_req', 'shots_per_rand_req', 'repeat'])
    assert len(r) == 15, f'expected 15 runs, found {len(r)}'
    tmpl = old.drop_duplicates('obs').set_index('obs')        # column layout of the existing PEC rows
    rep_col = next((c for c in ('rep', 'repetition', 'repeat', 'run') if c in all_df.columns), None)
    rows = []
    for k, (_, run) in enumerate(r.iterrows()):
        for o in OBS:
            row = tmpl.loc[o].to_dict(); row['obs'] = o
            row.update(measured_val=float(run[f'ev_{o}']), std=float(run[f'std_{o}']), sv_val=PEC_SV[o])
            for c, v in (('device', 'ibm_phoenix'), ('date', str(run.get('started_utc', ''))[:10]),
                         ('job_id', f"{run['batch']}/n{int(run['num_randomizations_req'])}"
                                    f"/s{int(run['shots_per_rand_req']) if pd.notna(run['shots_per_rand_req']) else 128}"
                                    f"/r{int(run['repeat'])}")):
                if c in all_df.columns: row[c] = v
            if rep_col: row[rep_col] = k
            rows.append(row)
    all_df = pd.concat([all_df[all_df.method != 'pec'], pd.DataFrame(rows)[all_df.columns]], ignore_index=True)
    all_df.to_csv(CSV, index=False)
    print(f'PEC rows replaced: {len(rows)} rows (15 runs x {len(OBS)} observables) written to {CSV}')

In [ ]:
df = pd.read_csv(CSV)
df = df[df.obs.isin(OBS)].copy()
df['rel_dev'] = (df.measured_val - df.sv_val) / df.sv_val
methods = [m for m in STYLE if STYLE[m] is not None and m in set(df.method)]

# Table 10: mean over the five observables of the signed relative deviation (repetitions averaged per observable)
per_obs = df.groupby(['method', 'obs']).agg(mean=('measured_val', 'mean'), sv=('sv_val', 'mean'),
                                            std=('std', 'first'), rel_dev=('rel_dev', 'mean'),
                                            sd_runs=('measured_val', lambda v: v.std(ddof=1)), n=('measured_val', 'size'))
table10 = (per_obs.groupby(level='method').rel_dev.mean() * 100).round(1).reindex(methods)
print('Table 10 - mean deviation from SV (%)'); print(table10.to_string(), '\n')

# Per-observable detail for the single-run methods
for m in ['qesem', 'pec']:
    if m in methods:
        d = per_obs.loc[m].reindex(OBS)
        d = d.assign(dev_pct=(d.rel_dev * 100).round(1))
        if m == 'pec':      # 15 runs: error bar = run-to-run SD
            print(f"PEC ({df[df.method == m].device.iloc[0]}, {int(d.n.iloc[0])} runs): per-observable mean "
                  f"{d.dev_pct.min()}% to {d.dev_pct.max()}%, run-to-run SD {d.sd_runs.min():.3f}-{d.sd_runs.max():.3f}")
            print(d[['mean', 'sv', 'sd_runs', 'dev_pct']].round(4).to_string(), '\n')
            key = next(c for c in ('job_id', 'rep', 'repetition', 'repeat', 'run') if c in df.columns)
            run_dev = df[df.method == 'pec'].groupby(key).rel_dev.mean() * 100
            print(f"PEC per run: mean {run_dev.mean():.1f}%, range {run_dev.min():.1f}% to {run_dev.max():.1f}% ({len(run_dev)} runs)\n")
        else:
            d = d.assign(z=((d['mean'] - d.sv) / d['std']).round(2))
            print(f"{m.upper()} ({df[df.method == m].device.iloc[0]}): per-observable {d.dev_pct.min()}% to {d.dev_pct.max()}%, "
                  f"std {d['std'].min():.3f}-{d['std'].max():.3f}")
            print(d[['mean', 'sv', 'std', 'dev_pct', 'z']].round(4).to_string(), '\n')

In [ ]:
x = np.arange(len(OBS))
fig, ax = plt.subplots(figsize=FIGSIZE)
for m in methods:
    label, colour, kind = STYLE[m]
    g = df[df.method == m].groupby('obs').measured_val
    mu = g.mean().reindex(OBS).values
    if kind == 'ref':
        # ax.plot(x, per_obs.loc[m].reindex(OBS)['sv'].values, '--', color=colour, lw=1.5, label=label)
        ax.plot(x, per_obs.loc[m].reindex(OBS)['sv'].values, '-', color=colour, lw=1.5, label=label)
    elif kind == 'band':
        ax.fill_between(x, g.min().reindex(OBS), g.max().reindex(OBS), color=colour, alpha=0.2, lw=0)
        # ax.plot(x, mu, color=colour, lw=1.5, label=label)
        ax.plot(x, mu, '--', marker=MARKER[m], ms=4, color=colour, lw=1.2, label=label)
    else:
        # QESEM: reported uncertainty of its single run; PEC: run-to-run SD over the 15 runs
        s = per_obs.loc[m].reindex(OBS)['sd_runs' if m == 'pec' else 'std'].values
        # ax.errorbar(x + POINT_OFFSET.get(m, 0), mu, yerr=s, fmt=POINT_MARKER.get(m, 'o'), color=colour, ms=6,
        #             capsize=3, lw=1.5, mfc='white' if m == 'pec' else colour, mew=1.5, label=label)
        ax.errorbar(x + POINT_OFFSET.get(m, 0), mu, yerr=s, fmt='--', marker=MARKER[m], ms=4, color=colour,
            capsize=3, lw=2, mfc=colour, mew=1.2, label=label)

ax.set_xticks(x, [rf'$\langle {o[0]}_{{q{o[1:]}}}\rangle$' for o in OBS])
ax.set_xlabel('Observable')
ax.set_ylabel('Expectation Value')
ax.set_ylim(*YLIM)
ax.axhline(0, color='0.85', lw=0.8, zorder=0)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
if SHOW_LEGEND:
    ax.legend(fontsize=8, loc='center left', bbox_to_anchor=(1.01, 0.5), frameon=False)
    # ax.legend(fontsize=8, loc='center right', bbox_to_anchor=(-0.1, 0.5), frameon=False)
fig.tight_layout()
fig.savefig(f'{OUT}.png', dpi=DPI, bbox_inches='tight')
fig.savefig(f'{OUT}.pdf', bbox_inches='tight')
plt.show()

The former optional cell that refreshed the PEC rows from the ibm_aachen job has been removed; the PEC rows now come from the 15 ibm_phoenix runs (cell "Replace the PEC rows").